In [1]:
import os
import random
import shutil
from pathlib import Path

dataset = Path(r"C:\Users\250442661\Downloads\project-6-at-2026-05-28-10-02-e37bcbec")  # change if needed
images_dir = dataset / "images"
labels_dir = dataset / "labels"

train_ratio = 0.8

image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

images = [p for p in images_dir.iterdir() if p.suffix.lower() in image_exts]
random.shuffle(images)

split_idx = int(len(images) * train_ratio)
train_images = images[:split_idx]
val_images = images[split_idx:]

for split, split_images in [("train", train_images), ("val", val_images)]:
    (images_dir / split).mkdir(parents=True, exist_ok=True)
    (labels_dir / split).mkdir(parents=True, exist_ok=True)

    for img_path in split_images:
        label_path = labels_dir / f"{img_path.stem}.txt"

        shutil.move(str(img_path), str(images_dir / split / img_path.name))

        if label_path.exists():
            shutil.move(str(label_path), str(labels_dir / split / label_path.name))
        else:
            print(f"Missing label for {img_path.name}")

In [ ]:
!pip install ultralytics

## Training

In [1]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=r"C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\data.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    name="my_yolo_model",

    # Early stopping
    patience=50  # stop if no improvement for 15 epochs
)

metrics = model.val()

Ultralytics 8.4.56  Python-3.14.4 torch-2.12.0+cpu CPU (Intel Core Ultra 5 125U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=my_yolo_model-3, nbs=64, nms=False, opset=None, opt

### Validation

In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
# Run prediction after training
from ultralytics import YOLO
from pathlib import Path
import cv2

# Load trained model
model = YOLO("runs/detect/my_yolo_model/weights/best.pt")

# Run prediction
results = model.predict(
    source=r"C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val",
    save=True,
    conf=0.25
)
scale = 4
# Directory to save crops
crop_dir = Path("cropped_boxes")
crop_dir.mkdir(exist_ok=True)

for result in results:
    # Read original image
    img = cv2.imread(result.path)

    # Loop through detections
    for i, box in enumerate(result.boxes):
        # Get bounding box coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Crop image
        crop = img[y1:y2, x1:x2]
        crop = cv2.resize(
            crop,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_LANCZOS4
        )

        # Get class name
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]

        # Save crop
        image_name = Path(result.path).stem
        crop_path = crop_dir / f"{image_name}_{cls_name}_{i}.tif"

        cv2.imwrite(str(crop_path), crop,
                    [cv2.IMWRITE_JPEG_QUALITY, 100])

        print(f"Saved: {crop_path}")


image 1/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val\0438236f-screenshot_017_frame_463.png: 352x640 1 kh, 77.4ms
image 2/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val\0ddb7dc4-screenshot_042_frame_802.png: 352x640 1 kh, 64.6ms
image 3/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val\0eeec62e-screenshot_020_frame_532.png: 352x640 1 kh, 60.7ms
image 4/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val\122ad6db-screenshot_026_frame_606.png: 352x640 1 kh, 62.7ms
image 5/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val\13043747-screenshot_021_frame_504.png: 352x640 1 kh, 70.1ms
image 6/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-10-02-e37bcbec\images\val\1e8568e6-screenshot_028_frame_575.png: 352x640 1 kh, 71.3ms
image 7/28 C:\Users\250442661\Downloads\Yolo\project-6-at-2026-05-28-

## Pipeline

### Extra cropping code (Don't run)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import tifffile
import numpy as np
# Load trained model
model = YOLO("runs/detect/my_yolo_model/weights/best.pt")

crop_dir = Path("cropped_boxes")
crop_dir.mkdir(exist_ok=True)

scale = 4

# Iterating prediction on frames from a tiff video
frames = tifffile.imread(r"C:\Users\250442661\Downloads\Yolo\Oprocessed_stack_81_3.20_1_2199.tif")
frame_count = 0
frame_number = 0
for frame in frames:

    frame_count += 1

    # Normalize if TIFF is 16-bit
    if frame.dtype != np.uint8:
        frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX)
        frame = frame.astype(np.uint8)

    # Convert grayscale to 3-channel
    if len(frame.shape) == 2:
        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

    # Run prediction on the frame
    results = model.predict(frame, conf=0.25)

    result = results[0]
    if len(result.boxes) > 0:
        print(f"Frame {frame_count}: detections found")
        if frame_number == 0:
            frame_number = frame_count
        else:
            frame_number = max(frame_number, frame_count)

    # Check if any detections were made
    if not result.boxes:
        continue

    for i, box in enumerate(result.boxes):
        # Get bounding box coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Crop image
        crop = frame[y1:y2, x1:x2]
        crop = cv2.resize(
            crop,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_LANCZOS4
        )

        # Get class name
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]

        # Save crop
        crop_path = crop_dir / f"frame_{frame_count}_{cls_name}_{i}.tif"
        cv2.imwrite(str(crop_path), crop,
                    [cv2.IMWRITE_JPEG_QUALITY, 100])

        print(f"Saved: {crop_path}")
    


0: 352x640 (no detections), 42.8ms
Speed: 1.6ms preprocess, 42.8ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 36.0ms
Speed: 1.4ms preprocess, 36.0ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 36.8ms
Speed: 1.2ms preprocess, 36.8ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 33.8ms
Speed: 1.1ms preprocess, 33.8ms inference, 0.4ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 37.6ms
Speed: 1.3ms preprocess, 37.6ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 35.4ms
Speed: 1.1ms preprocess, 35.4ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 35.8ms
Speed: 1.4ms preprocess, 35.8ms inference, 0.4ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 38.3ms
Speed: 1.2ms preprocess, 38.3ms i

### Detections and Group formation

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import tifffile
import numpy as np

model = YOLO("runs/detect/my_yolo_model-3/weights/best.pt")

crop_dir = Path("cropped_boxes")
crop_dir.mkdir(exist_ok=True)

scale = 4
MAX_FRAME_GAP = 10
IOU_THRESHOLD = 0.5   # increase to 0.3 if you want stricter overlap

frames = tifffile.imread(
    r"C:\Users\250442661\Downloads\Yolo\Oprocessed_stack_81_3.20_1_2199.tif"
)

output_tiff_path = "annotated_detection_stack.tif"

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area

    if union == 0:
        return 0

    return inter_area / union


groups = []
next_group_id = 1
annotated_frames = []

for frame_count, frame in enumerate(frames, start=1):

    if frame.dtype != np.uint8:
        frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX)
        frame = frame.astype(np.uint8)

    if len(frame.shape) == 2:
        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

    annotated_frame = frame.copy()

    results = model.predict(frame, conf=0.25)
    result = results[0]

    if len(result.boxes) == 0:
        annotated_frames.append(annotated_frame)
        continue

    print(f"Frame {frame_count}: detections found")

    for i, box in enumerate(result.boxes):

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        current_box = [x1, y1, x2, y2]

        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]

        conf = float(box.conf[0])

        matched_group = None
        best_iou = 0

        for group in groups:
            last_box = group["last_box"]
            last_frame = group["last_frame"]

            frame_gap = frame_count - last_frame

            if frame_gap <= MAX_FRAME_GAP:
                iou = compute_iou(current_box, last_box)

                if iou > best_iou and iou >= IOU_THRESHOLD:
                    best_iou = iou
                    matched_group = group

        if matched_group is None:
            matched_group = {
                "group_id": next_group_id,
                "last_box": current_box,
                "last_frame": frame_count,
                "detections": []
            }
            groups.append(matched_group)
            next_group_id += 1

        matched_group["last_box"] = current_box
        matched_group["last_frame"] = frame_count
        matched_group["detections"].append({
            "frame": frame_count,
            "box": current_box,
            "class": cls_name,
            "iou": best_iou
        })

        group_id = matched_group["group_id"]

        # Draw bounding box
        cv2.rectangle(
            annotated_frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        label = f"G{group_id} {cls_name} {conf:.2f}"

        cv2.putText(
            annotated_frame,
            label,
            (x1, max(y1 - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
            cv2.LINE_AA
        )

        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        crop = cv2.resize(
            crop,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_LANCZOS4
        )

        group_dir = crop_dir / f"group_{group_id}"
        group_dir.mkdir(exist_ok=True)

        crop_path = group_dir / f"frame_{frame_count}_{cls_name}_{i}.tif"

        # Uncomment to save crops
        # cv2.imwrite(str(crop_path), crop)
        # print(f"Saved: {crop_path}")
    
    annotated_frames.append(annotated_frame)

# Convert list to TIFF stack
annotated_frames = np.stack(annotated_frames, axis=0)

# Convert BGR to RGB before saving, because OpenCV uses BGR
annotated_frames_rgb = annotated_frames[:, :, :, ::-1]

tifffile.imwrite(
    output_tiff_path,
    annotated_frames_rgb,
    photometric="rgb"
)

print(f"\nSaved annotated TIFF stack: {output_tiff_path}")

print("\nDetection groups created:")
for group in groups:
    print(
        f"Group {group['group_id']}: "
        f"{len(group['detections'])} detections, "
        f"frames {group['detections'][0]['frame']} to {group['detections'][-1]['frame']}"
    )


0: 352x640 (no detections), 150.3ms
Speed: 2.3ms preprocess, 150.3ms inference, 1.0ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 44.3ms
Speed: 3.9ms preprocess, 44.3ms inference, 0.5ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 41.4ms
Speed: 1.4ms preprocess, 41.4ms inference, 0.4ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 47.4ms
Speed: 2.0ms preprocess, 47.4ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 85.7ms
Speed: 1.6ms preprocess, 85.7ms inference, 0.4ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 78.4ms
Speed: 2.5ms preprocess, 78.4ms inference, 0.4ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 62.0ms
Speed: 2.4ms preprocess, 62.0ms inference, 0.6ms postprocess per image at shape (1, 3, 352, 640)

0: 352x640 (no detections), 64.9ms
Speed: 1.8ms preprocess, 64.9ms

### Amplitude Computation

In [ ]:
import cv2
import numpy as np

def find_highest_point(
    image,
    threshold=150,
    pixel_to_mm=0.212,
    return_mask=False
):
    """
    Find the highest point of the largest bright component in an image.

    Parameters
    ----------
    image : np.ndarray
        Input grayscale or RGB image.
    threshold : int
        Threshold for bright pixels.
    pixel_to_mm : float
        Pixel-to-mm conversion factor.
    return_mask : bool
        If True, also returns the binary mask.

    Returns
    -------
    dict or None
        {
            "x": x coordinate,
            "y": y coordinate (OpenCV coordinates),
            "y_bottom": bottom-origin y,
            "amplitude_mm": amplitude in mm
        }

        Returns None if no valid component is found.
    """

    # Convert to grayscale if needed
    if image.ndim == 3:
        if image.shape[-1] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        elif image.shape[-1] == 4:
            image = cv2.cvtColor(image, cv2.COLOR_RGBA2GRAY)

    height = image.shape[0]

    # Convert to uint8 if necessary
    if image.dtype != np.uint8:
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Threshold
    _, binary = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)
    binary = binary.astype(np.uint8)

    # Connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8
    )

    if num_labels <= 1:
        return None

    # Largest bright component
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    main_mask = labels == largest_label

    ys, xs = np.where(main_mask)

    if len(xs) == 0:
        return None

    # Highest point
    top_row = ys.min()
    top_x = int(np.mean(xs[ys == top_row]))

    y_bottom = height - top_row
    amplitude_mm = y_bottom * pixel_to_mm


    result = {
        "x": top_x,
        "y": int(top_row),
        "y_bottom": int(y_bottom),
        "amplitude_mm": float(amplitude_mm),
    }

    if return_mask:
        result["mask"] = binary

    return result

In [4]:
# Processing the groups to calculate amplitude on boxes of groups with more than 40 detections
for group in groups:
    if len(group['detections']) > 40:
        amplitudes = []
        frame_number = group['detections'][0]['frame']
        print(f"Processing Group {group['group_id']} with {len(group['detections'])} detections")
        # Here you can add your amplitude calculation logic for the group
        for detection in group['detections']:
            frame_num = detection['frame']
            box = detection['box']
            x1, y1, x2, y2 = box
            # Extract the corresponding frame from the original frames
            frame = frames[frame_num - 1]  # Adjust for zero-based index
            # Crop the box from the frame
            crop = frame[y1:y2, x1:x2]
            # Calculate amplitude (example: max pixel value in the crop)
            result = find_highest_point(crop)
            amplitudes.append(result['amplitude_mm'] if result else None)
        
        # preparing the amplitude graph against frame number for the group and save it
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 5))
        plt.plot([d['frame'] for d in group['detections']], amplitudes, marker='o')
        plt.xlabel('Frame Number')
        plt.ylabel('Amplitude (mm)')
        plt.title(f'Amplitude Variation for Group {group["group_id"]}')
        plt.grid(True)
        plt.savefig(f'group_{group["group_id"]}_amplitude.png')
        plt.close()

Processing Group 11 with 45 detections
Processing Group 13 with 48 detections
Processing Group 17 with 63 detections
Processing Group 38 with 120 detections
Processing Group 75 with 60 detections
Processing Group 77 with 41 detections
Processing Group 78 with 60 detections
Processing Group 79 with 350 detections
Processing Group 86 with 239 detections
Processing Group 97 with 56 detections
Processing Group 98 with 50 detections
